# ការកក់សណ្ឋាគារជាមួយ Middleware សម្រាប់សមាជិកអាទិភាព

This notebook demonstrates **function-based middleware** using the Microsoft Agent Framework. We build upon the conditional workflow example by adding a middleware layer that gives priority members special privileges.

## អ្វីដែលអ្នកនឹងរៀន៖
1. **Function-Based Middleware**: ឆ្លាមិងកែប្រែលទ្ធផលរបស់មុខងារ
2. **Context Access**: អាន និងកែប្រែ `context.result` បន្ទាប់ពីការដំណើរការ
3. **Business Logic Implementation**: អត្ថប្រយោជន៍សម្រាប់សមាជិកអាទិភាព
4. **Result Override**: ប្ដូរលទ្ធផលមុខងារតាមស្ថានភាពអ្នកប្រើ
5. **Same Workflow, Different Outcomes**: ការផ្លាស់ប្តូរនៃអាកប្បកិរិយាដោយ middleware

## ស្ថាបត្យកម្ម Workflow ជាមួយ Middleware:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## ភាពខុសគ្នាសំខាន់ពី workflow លក្ខខណ្ឌ:

**ដោយគ្មាន Middleware** (14-conditional-workflow.ipynb):
- Paris has no rooms → បញ្ជូនទៅ alternative_agent

**ជាមួយ Middleware** (this notebook):
- អ្នកប្រើធម្មតា + Paris → គ្មានបន្ទប់ → បញ្ជូនទៅ alternative_agent
- សមាជិកអាទិភាព + Paris → 🌟 Middleware ប្ដូរលទ្ធផល! → មានបន្ទប់ → បញ្ជូនទៅ booking_agent

## តម្រូវការមុន:
- បានដំឡើង Microsoft Agent Framework
- មានការយល់ដឹងអំពី workflow លក្ខខណ្ឌ (សូមមើល 14-conditional-workflow.ipynb)
- GitHub token ឬ OpenAI API key
- យល់ដឹងមូលដ្ឋានអំពីគំរូ middleware


In [1]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    ChatMessage,
    FunctionInvocationContext,
    Role,
    WorkflowBuilder,
    WorkflowContext,
    ai_function,
    executor,
)

# 🤖 GitHub Models or OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

✅ All imports successful!


## ជំហាន 1: កំណត់ម៉ូដែល Pydantic សម្រាប់លទ្ធផលដែលមានរចនាសម្ព័ន្ធ

ម៉ូដែលទាំងនេះកំណត់ **schema** ដែលភ្នាក់ងារ​នឹងត្រឡប់វិញ។ យើងបានបន្ថែមវាល `priority_override` ដើម្បីតាមដានពេលដែល middleware កែប្រែលទ្ធផលភាពអាចប្រើបាន។


In [2]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    priority_override: bool = False  # 🆕 NEW! Tracks if middleware overrode the result


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

✅ Pydantic models defined:
   - BookingCheckResult (availability check with priority_override)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)


## ជំហាន 2: កំណត់មូលដ្ឋានទិន្នន័យសមាជិកអាទិភាព

សម្រាប់ការបង្ហាញនេះ យើងនឹងប្រើមូលដ្ឋានទិន្នន័យសមាជិកអាទិភាពជាគំរូ។ ក្នុងបរិបទផលិតកម្ម វានឹងស្វែងរកពីមូលដ្ឋានទិន្នន័យពិត ឬ API។

**សមាជិកអាទិភាព:**
- `alice@example.com` - សមាជិក VIP
- `bob@example.com` - សមាជិកព្រីមៀម  
- `priority_user` - គណនីសាកល្បង


In [3]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

✅ Priority members database created
   Priority members: 3 users


## ជំហាន 3: បង្កើតឧបករណ៍កក់សណ្ឋាគារ

ដូចដដែលនឹងចរន្តការងារមានលក្ខខណ្ឌ, ប៉ុន្តែឥឡូវនេះវានឹងត្រូវបានរារាំងដោយ middleware!


In [4]:
@ai_function(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @ai_function decorator")

✅ hotel_booking tool created with @ai_function decorator


## ជំហានទី 4: 🌟 បង្កើត Middleware សម្រាប់ពិនិត្យអាទិភាព (មុខងារស្នូល!)

នេះគឺជា **មុខងារស្នូល** នៃកំណត់ត្រានេះ។ Middleware នេះ៖

1. **ចាប់យក** ការហៅអនុគមន៍ hotel_booking
2. **អនុវត្ត** អនុគមន៍ជាធម្មតា ដោយហៅ `next(context)`
3. **ពិនិត្យ** លទ្ធផលក្នុង `context.result`
4. **ប្តូរជំនួស** លទ្ធផល ប្រសិនបើអ្នកប្រើមានអាទិភាព និងមិនមានបន្ទប់ទំនេរ
5. **សងត្រឡប់** លទ្ធផលដែលបានកែប្រែទៅកាន់ភ្នាក់ងារ

**លំនាំសំខាន់:**
```python
async def my_middleware(context, next):
    await next(context)  # អនុវត្តមុខងារ
    # ឥឡូវ context.result មានលទ្ធផលនៃមុខងារ
    if some_condition:
        context.result = new_value  # ជំនួស!
```


In [5]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

✅ priority_check_middleware created
   - Intercepts hotel_booking function
   - Overrides availability for priority members


## ជំហានទី 5: កំណត់មុខងារ​លក្ខខណ្ឌ​សម្រាប់​ការ​បញ្ជូន

មុខងារ​លក្ខខណ្ឌដូចគ្នានៃលំហូរការងារ​លក្ខខណ្ឌ - ពួកវាពិនិត្យលទ្ធផលដែលមានរចនាសម្ព័ន្ធ ដើម្បីកំណត់ការបញ្ជូន។


In [6]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

✅ Condition functions defined


## ជំហាន 6: បង្កើត Custom Display Executor

Executor ដូចដើម - បង្ហាញលទ្ធផលចុងក្រោយនៃដំណើរការងារ។


In [7]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

✅ display_result executor created


## ជំហាន 7: ផ្ទុកអថេរបរិស្ថាន

កំណត់រចនាសម្ព័ន្ធអតិថិជន LLM (GitHub Models or OpenAI).


In [8]:
# Load environment variables
load_dotenv()

# Check for GitHub Models or OpenAI
chat_client = OpenAIChatClient(base_url=os.environ.get(
    "GITHUB_ENDPOINT"), api_key=os.environ.get("GITHUB_TOKEN"), model_id="gpt-4o")


## ជំហានទី 8: បង្កើតភ្នាក់ងារ AI ជាមួយ Middleware

**ចំណុចខុសគ្នាសំខាន់:** ពេលបង្កើត availability_agent, យើងផ្តល់ប៉ារ៉ាម៉ែត្រ `middleware`!

នេះជា​របៀប​ដែល​យើងបញ្ចូល priority_check_middleware ទៅក្នុង pipeline នៃការហៅមុខងារ​របស់ agent។


In [9]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        response_format=BookingCheckResult,
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        response_format=AlternativeResult,
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        response_format=BookingConfirmation,
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## ជំហាន ៩: បង្កើតលំហូរការងារ

រចនាសម្ព័ន្ធលំហូរការងារដូចមុន - ការបញ្ជូនតាមលក្ខខណ្ឌដែលផ្អែកលើភាពអាចប្រើបាន។


In [10]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder()
    .set_start_executor(availability_agent)
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## ជំហាន 10: ករណីសាកល្បង 1 - អ្នកប្រើធម្មតា នៅប៉ារីស (គ្មានការជំនួស)

អ្នកប្រើធម្មតា ព្យាយាមកក់នៅប៉ារីស → គ្មានបន្ទប់ → ផ្ញើទៅ alternative_agent


In [11]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## ជំហាន 11: ករណីសាកល្បង 2 - 🌟 អ្នកប្រើអាទិភាពនៅ Paris (ជាមួយ Override!)

សមាជិកអាទិភាពព្យាយាមកក់ Paris → នៅដើម គ្មានបន្ទប់ → 🌟 Middleware ជំនួស! → ទៅកាន់ booking_agent

**នេះជាការបង្ហាញសំខាន់នៃអំណាចរបស់ middleware!**


In [12]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; 
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## ជំហាន 12: ករណីសាកល្បង 3 - អ្នកប្រើអាទិភាព នៅ Stockholm (មានរួចហើយ)

អ្នកប្រើអាទិភាព ព្យាយាម Stockholm → មានបន្ទប់ទំនេរ → មិនចាំបាច់ធ្វើការជំនួស → បញ្ជូនទៅ booking_agent

នេះបង្ហាញថា ស្រទាប់មធ្យម សកម្មតែពេលដែលចាំបាច់!


In [13]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; 
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## ចំណុច​សំខាន់ និង គំនិត​អំពី Middleware

### ✅ អ្វី​ដែលអ្នក​បាន​រៀន៖

#### **1. តម្រង Middleware ផ្អែកលើ​មុខងារ**

Middleware ចាប់ស្នងការហៅមុខងារ ដោយប្រើមុខងារ async ងាយៗ៖

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # មុនការប្រតិបត្តិរបស់មុខងារ
    print("Intercepting...")
    
    # អនុវត្តមុខងារ
    await next(context)
    
    # បន្ទាប់ពីការប្រតិបត្តិរបស់មុខងារ - ពិនិត្យលទ្ធផល
    if context.result:
        # កែប្រែលទ្ធផល ប្រសិនបើចាំបាច់
        context.result = modified_value
```

#### **2. ការចូលដំណើរការ Context និង ការកែប្រែលទ្ធផល**

- `context.function` - ចូលដល់មុខងារដែលត្រូវបានហៅ
- `context.arguments` - អាន arguments របស់មុខងារ
- `context.kwargs` - ចូលដល់ប៉ារ៉ាម៉ែត្របន្ថែម
- `await next(context)` - អនុវត្តមុខងារ
- `context.result` - អាន/កែប្រែលទ្ធផលដែលមុខងារបានបញ្ចេញ

#### **3. ការអនុវត្តលក្ខខណ្ឌអាជីវកម្ម**

Middleware របស់យើងអនុវត្តអត្ថប្រយោជន៍​សមាជិក​ចំណាត់ថ្នាក់អាទិភាព៖
- **អ្នកប្រើធម្មតា**: មិនមានការផ្លាស់ប្តូរ ដំណើរការតាមប្រព័ន្ធស្ដង់ដារ
- **អ្នកប្រើអាទិភាព**: ប្ដូរ "no availability" → "available"
- **Conditional logic**: ប្ដូរតែពេលចាំបាច់

#### **4. ដំណើរការដដែល ប៉ុន្តែលទ្ធផលខុសគ្នា**

អំណាចនៃ middleware:
- ✅ មិនមានការផ្លាស់ប្តូរចំពោះរចនាសម្ព័ន្ធដំណើរការ
- ✅ មិនមានការផ្លាស់ប្តូរដល់មុខងារ​ឧបករណ៍
- ✅ មិនមានការផ្លាស់ប្តូរទៅលើវិធីបង្វិលតាមលក្ខខណ្ឌ
- ✅ គ្រាន់តែ middleware → អាកប្បកិរិយាផ្សេង!

### 🚀 ការប្រើប្រាស់ក្នុងពិភពពិត៖

1. **VIP/Premium Features**
   - ប្ដូរកំណត់អត្រាដល់អ្នកប្រើ premium
   - ផ្តល់ការចូលដំណើរការអាទិភាពទៅប្រភពធនធាន
   - បើកលក្ខណៈ premium ដោយឆ្លើយតបទាន់ពេល

2. **A/B Testing**
   - បញ្ជូនអ្នកប្រើទៅកាន់ការអនុវត្តន៍ផ្សេងៗ
   - សាកល្បងលក្ខណៈថ្មីជាមួយអ្នកប្រើជាក់លាក់
   - ចាក់ចេញលក្ខណៈជាជំហ៊ាន

3. **សន្តិសុខ និងការអនុលោម**
   - ត្រួតពិនិត្យការហៅមុខងារ
   - រារាំងប្រតិបត្តិការដែលមានភាពសំងាត់
   - អនុវត្តច្បាប់អាជីវកម្ម

4. **បង្កើនប្រសិទ្ធភាព**
   - ផ្ទុកលទ្ធផលក្នុង cache សម្រាប់អ្នកប្រើជាក់លាក់
   - លោតលើប្រតិបត្តិការរត់ថ្លៃពេលអាច
   - ចែកចាយធនធានដោយឆ្លើយតប

5. **ការចាប់យកកំហុស និងការព្យាយាមម្តងទៀត**
   - ចាប់ និង ដោះស្រាយកំហុសយ៉ាងរាបស្មើ
   - អនុវត្ត​យុទ្ធសាស្រ្ត​ព្យាយាមម្តងទៀត
   - បិទទៅកាន់ការអនុវត្តជំនួស

6. **កត់ត្រា និងតាមដាន**
   - តាមដានពេលវេលាប្រតិបត្តិមុខងារ
   - កត់ត្រាប៉ារ៉ាម៉ែត្រ និងលទ្ធផល
   - តាមដានគំរូការប្រើប្រាស់

### 🔑 ការប្រៀបធៀបចម្បងពី Decorators:

| លក្ខណៈ | Decorator | Middleware |
|---------|-----------|------------|
| **ដែនកំណត់** | មុខងារតែមួយ | មុខងារទាំងអស់នៅក្នុង agent |
| **ភាពបត់បែន** | ថេរនៅពេលកំណត់ | បត់បែននៅពេលរត់កម្មវិធី |
| **Context** | មានកំណត់ | បរិបទ agent ពេញលេញ |
| **សមាសធាតុ** | Decorator ច្រើន | បណ្តាញ Middleware |
| **Agent-Aware** | ទេ | បាទ (ចូលដល់ស្ថានភាព agent) |

### 📚 ពេលណាដែលគួរប្រើ Middleware:

✅ **ប្រើ middleware នៅពេល:**
- អ្នកត្រូវប្ដូរអាកប្បកិរិយាដោយផ្អែកលើស្ថានភាពអ្នកប្រើ/សម័យ
- អ្នកចង់អនុវត្តតុល្យការ​ទៅលើមុខងារច្រើន
- អ្នកត្រូវការចូលដល់បរិបទជាថ្នាក់ agent
- អ្នកកំពុងអនុវត្តបញ្ហាកាត់ផ្សេង (កត់ត្រា, ការផ្ទៀងផ្ទាត់, ល។)

❌ **កុំប្រើ middleware នៅពេល:**
- ការផ្ទៀងផ្ទាត់បញ្ចូលពីងាយៗ (ប្រើ Pydantic)
- លក្ខខណ្ឌពិសេសសម្រាប់មុខងារ (ទុកនៅក្នុងមុខងារ)
- ការផ្លាស់ប្តូរម្តងតែមួយ (ប្ដូរមុខងារម្តង)

### 🎓 គំរូកម្រិតខ្ពស់:

```python
# មីឌលវែរ​ច្រើន (លំដាប់អនុវត្តមានសារៈសំខាន់!)
middleware=[
    logging_middleware,      # កត់ត្រាមុន
    auth_middleware,         # បន្ទាប់មកពិនិត្យការផ្ទៀងផ្ទាត់
    cache_middleware,        # បន្ទាប់មកពិនិត្យកែស
    rate_limit_middleware,   # បន្ទាប់មកដាក់កំណត់អត្រា
    priority_check_middleware  # ចុងក្រោយពិនិត្យអាទិភាព
]

# ការអនុវត្តមីឌលវែរ​តាមលក្ខខណ្ឌ
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # កែប្រែលទ្ធផល
    else:
        # រំលងការអនុវត្តទាំងស្រុង
        context.result = cached_value
```

### 🔗 គំនិត​ពាក់ព័ន្ធ:

- **Agent Middleware**: ចាប់ស្នងការហៅ agent.run()
- **Function Middleware**: ចាប់ស្នងការហៅមុខងារឧបករណ៍ (អ្វីដែលយើងបានប្រើ!)
- **Middleware Pipeline**: ខ្សែសង្វាក់នៃ middleware ដែលរត់តាមលំដាប់
- **Context Propagation**: ផ្ញើស្ថានភាពឆ្លងកាត់ខ្សែ middleware


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator). ខណៈពេលយើងខិតខំសម្រាប់ភាពត្រឹមត្រូវ សូមយកចិត្តទុកដាក់ថាការបកប្រែដោយស្វ័យប្រវត្តិក៏អាចមានកំហុស ឬភាពមិនត្រឹមត្រូវ។ ឯកសារ​ដើម​នៅក្នុង​ភាសា​ដើម​គួរត្រូវបានចាត់ទុកថាជា​ប្រភព​ដែលមាន​អាទិភាព។ សម្រាប់ព័ត៌មានសំខាន់ៗ យើងសូមណែនាំឱ្យប្រើការបកប្រែដោយអ្នកបកប្រែមនុស្សវិជ្ជាជីវៈ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសណាមួយដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
